In [1]:
!pip install poola

#WARNING:

This pipeline requires you to have these 6 files in the same directory as the notebook:

counts_with_mismatches_CBE_activity_A375_DO_PIC.csv\
counts_with_mismatches_CBE_activity_A375_LIN_SCH.csv\
counts_with_mismatches_CBE_activity_HT29_DO_PIC.csv\
counts_with_mismatches_CBE_activity_HT29_LIN_SCH.csv\
MC_readcouns.csv \
Base Editing Screens Samplesheet.xlsx

In [2]:
"""
Base-editing screen analysis pipeline -- CoelhoNatGen and dt studies (dt is out latest study).

Pipeline overview
------------------
1.  Raw per-guide read counts are read in and reshaped into one wide table
    per study: `process_CoelhoNatGen` for the CoelhoNatGen study, `process_dt` for the dt
    study.
2.  Read counts are log2-normalized and pDNA-filtered via `poola`
    (`poola_normalize_and_filter`). From this point on, every count column
    in the pipeline holds LOG2 values, not raw counts.
3.  For each comparison defined in the samplesheet's 'Comparisons' sheet
    (a numerator condition vs. a denominator condition), a log2
    fold-change is computed. See `compute_log2fc_and_zscore` for exactly
    what this involves and why it's a subtraction.
4.  Each fold-change column is converted to a Z-score, using the
    Intergenic-control guides in that column as the "no effect" null
    distribution (mean/std).
5.  Replicate columns (`_RepA`, `_RepB`, ...) are collapsed to one value
    per condition by `collapse_replicates_min`.
6.  `process_df` is the single public entry point: it dispatches to the
    CoelhoNatGen or dt pipeline based on `study`, and writes the resulting CSVs.

`process_df` is the only function most callers need. Everything else in
this file is a building block it (or `process_dt`) calls internally.
"""

import re

import numpy as np
import pandas as pd
from functools import lru_cache
from poola import core as pool


# =============================================================================
# Configuration -- every path, column list, and lookup table lives here so
# they're easy to find and update without hunting through function bodies.
# =============================================================================

# --- File locations ---------------------------------------------------------
SAMPLESHEET_PATH = 'Base Editing Screens Samplesheet.xlsx'
dt_HT29_DO_PATH = 'counts_with_mismatches_CBE_activity_HT29_DO_PIC.csv'
dt_HT29_LIN_PATH = 'counts_with_mismatches_CBE_activity_HT29_LIN_SCH.csv'

# --- Shared column names -----------------------------------------------------
META_COLS = ['Guide', 'Gene', 'Editor']

# --- CoelhoNatGen study: column-renaming rules -----------------------------------------
# Cell line codes as they appear in the raw condition names -> underscored
# display form used in the output columns.
CELL_LINE_RENAMES = {
    r'^HT29': 'HT_29',
    r'^PC9': 'PC_9',
    r'^H23': 'H_23',
    r'^MHHES1': 'MHHES_1',
}

# Raw replicate tags -> display replicate labels.
REPLICATE_RENAMES = {
    'exp1': 'RepA',
    'exp2': 'RepB',
    'exp3': 'RepC',
}

# Genes treated as controls when pulling QC/control guide annotations for CoelhoNatGen.
CoelhoNatGen_CONTROL_GENES = ['Panlethal splice donor', 'Intergenic control', 'Non-targeting control']

# --- dt study: which columns to keep from each cell-line/treatment file -----
A375_DO_COLS = ['Construct Barcode', 'A375_RDA270_CP2165_PIC_RepA', 'A375_RDA270_CP2165_DO_RepB',
                'A375_RDA270_CP2165_DO_RepA', 'A375_RDA270_CP2165_PIC_RepB', 'pDNA_CP2165']
A375_LIN_COLS = ['Construct Barcode', 'A375_RDA270_CP2165_SCH_RepB', 'pDNA_CP2165',
                  'A375_RDA270_CP2165_LIN_RepB', 'A375_RDA270_CP2165_SCH_RepA', 'A375_RDA270_CP2165_LIN_RepA']
HT29_DO_COLS = ['Construct Barcode', 'HT29_RDA270_CP2165_PIC_RepA', 'HT29_RDA270_CP2165_DO_RepB',
                'HT29_RDA270_CP2165_DO_RepA', 'HT29_RDA270_CP2165_PIC_RepB', 'pDNA_CP2165']
HT29_LIN_COLS = ['Construct Barcode', 'HT29_RDA270_CP2165_SCH_RepB', 'pDNA_CP2165',
                  'HT29_RDA270_CP2165_LIN_RepB', 'HT29_RDA270_CP2165_SCH_RepA', 'HT29_RDA270_CP2165_LIN_RepA']

# --- comparison name -> output-filename suffix ------------------------------
# Used by `compute_lfc_zscore_and_save` to name its three output CSVs.
CoelhoNatGen_COMPARISON_SUFFIXES = {
    'CoelhoNatGen_processed': ''
}
dt_COMPARISON_SUFFIXES = {
    'dt': ''
}

# --- what `process_df` will accept -------------------------------------------
VALID_STUDIES = {'CoelhoNatGen', 'dt'}
VALID_COMPARISONS = {
    'CoelhoNatGen_processed', 'dt'
}


# =============================================================================
# Generic helpers
# =============================================================================

@lru_cache(maxsize=None)
def _read_samplesheet(sheet_name: str) -> pd.DataFrame:
    """Cached read of one sheet from SAMPLESHEET_PATH, so repeated calls in
    the same run don't re-parse the same Excel file."""
    return pd.read_excel(SAMPLESHEET_PATH, sheet_name=sheet_name)


def reorder_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Move Guide, Gene, Editor to the front of the dataframe."""
    cols = df.columns.tolist()
    new_order = META_COLS + [col for col in cols if col not in META_COLS]
    return df[new_order]


def poola_normalize_and_filter(df: pd.DataFrame, pdna_col: str, meta_cols=None) -> pd.DataFrame:
    """
    Log2-normalize every read-count column via poola, then drop guides whose
    pDNA (plasmid) abundance is more than 3 standard deviations below the
    mean -- these are guides that were under-represented in the original
    plasmid pool, so their downstream fold-changes aren't trustworthy.

    IMPORTANT: every count column returned by this function is a LOG2 value,
    not a raw read count. All downstream fold-change math assumes this.
    """
    df = df.copy()
    meta_cols = meta_cols or META_COLS
    read_cols = [col for col in df.columns if col not in meta_cols]

    lognorm_df = pool.lognorm_columns(reads_df=df, columns=read_cols)
    return pool.filter_pdna(lognorm_df=lognorm_df, pdna_cols=[pdna_col], z_low=-3)


# =============================================================================
# Statistics: log2 fold-change and Z-scoring
#
# This is the analytical core of the pipeline -- everything else exists to
# get a table into the shape this function expects.
# =============================================================================

def compute_log2fc_and_zscore(log_normalized_counts: pd.DataFrame, comparison_name: str):
    """
    For every comparison listed in the samplesheet, compute a log2
    fold-change column and a Z-scored version of it.

    Why this is a SUBTRACTION, not a division
    -------------------------------------------
    `log_normalized_counts` must already have been through
    `poola_normalize_and_filter`, so every count column already holds LOG2
    values -- not raw read counts.

    An ordinary fold-change is `numerator_reads / denominator_reads`. Since
    log2(a / b) == log2(a) - log2(b), computing a fold-change from values
    that are *already* log2-transformed is just a subtraction:

        log2_fold_change = log2(numerator_reads) - log2(denominator_reads)
                          = log_reads_numerator   - log_reads_denominator

    So below, `numerator_column` / `denominator_column` are condition
    *names* pulled from the samplesheet (e.g. 'HT_29_Sotor_RepA'), and
    `log_reads_numerator` / `log_reads_denominator` are the corresponding
    already-log2 values. The "numerator/denominator" naming is a holdover
    from the un-logged fold-change formula above -- mathematically correct
    here, just easy to misread as "these get divided".

    Z-scoring against Intergenic controls
    ----------------------------------------
    Each fold-change column is then converted to a Z-score using ONLY the
    guides targeting 'Intergenic control' regions to estimate the null
    mean/std, since those guides are assumed to have no real biological
    effect. Every guide's fold-change is expressed relative to that
    "no effect" baseline.

    Parameters
    ----------
    log_normalized_counts : pd.DataFrame
        One row per guide, with Guide/Gene/Editor meta-columns plus one
        already-log2-normalized column per condition.
    comparison_name : str
        A 'ComparisonName' value from the samplesheet's 'Comparisons'
        sheet. If it doesn't match any row, ALL rows in the sheet are used
        instead -- this mirrors pre-existing behavior (an unrecognized name
        silently means "run every comparison"), not something new.

    Returns
    -------
    (log2fc_df, zscore_df) : tuple of pd.DataFrame
        Both share the same Guide/Gene/Editor meta-columns, plus one column
        per comparison ('ColumnName' from the samplesheet).

    Side effect
    -----------
    Columns of `log_normalized_counts` that get used as a numerator or
    denominator are coerced to numeric IN PLACE (mutating the caller's
    dataframe). This matches the original implementation; flagged here
    because it's a non-obvious side effect for a function that looks
    read-only.
    """
    comparisons = _read_samplesheet('Comparisons')
    if comparison_name in comparisons['ComparisonName'].unique():
        comparisons = comparisons[comparisons['ComparisonName'] == comparison_name]

    log2fc_df = log_normalized_counts[META_COLS].copy()
    zscore_df = log2fc_df.copy()

    # --- log2 fold-change: numerator minus denominator (see docstring) -----
    for _, row in comparisons.iterrows():
        numerator_column = row['ConditionNumerator']
        denominator_column = row['ConditionDenominator']
        comparison_column_name = row['ColumnName']

        if numerator_column in log_normalized_counts.columns and denominator_column in log_normalized_counts.columns:
            log_normalized_counts[numerator_column] = pd.to_numeric(
                log_normalized_counts[numerator_column], errors='coerce'
            )
            log_normalized_counts[denominator_column] = pd.to_numeric(
                log_normalized_counts[denominator_column], errors='coerce'
            )
            log_reads_numerator = log_normalized_counts[numerator_column]
            log_reads_denominator = log_normalized_counts[denominator_column]

            log2fc_df[comparison_column_name] = log_reads_numerator - log_reads_denominator

    # --- Z-score each fold-change column against Intergenic controls -------
    for comparison_column_name in comparisons['ColumnName']:
        if comparison_column_name in log2fc_df.columns:
            control_fold_changes = log2fc_df.loc[
                log2fc_df['Gene'] == 'Intergenic control', comparison_column_name
            ].dropna()
            control_mean = control_fold_changes.mean()
            control_std = control_fold_changes.std()

            zscore_df[comparison_column_name] = (
                log2fc_df[comparison_column_name] - control_mean
            ) / control_std

    return log2fc_df, zscore_df


# Backward-compatible alias for the old name, in case anything outside this
# file still calls `LFC_Z(...)` directly.
LFC_Z = compute_log2fc_and_zscore


def collapse_replicates_min(df: pd.DataFrame) -> pd.DataFrame:
    """
    Collapse per-replicate columns (suffixed _RepA, _RepB, ...) down to one
    column per condition.

    When a condition has 2+ replicates, the value kept for each guide is
    whichever replicate's value has the SMALLEST magnitude (closest to
    zero) -- i.e. the more conservative of the replicates. A guide only
    ends up looking like a strong hit here if both/all replicates agree it
    is one; a replicate with an outsized effect gets overridden by a
    replicate closer to baseline.

    When a condition has only 1 replicate, that single value is used as-is.
    """
    rep_dict = {}
    for col in df.columns:
        match = re.match(r'(.*)_(Rep[A-Z])$', col)
        if match:
            condition, rep = match.group(1), match.group(2)
            rep_dict.setdefault(condition, {})[rep] = col

    meta_cols = [col for col in META_COLS if col in df.columns]
    collapsed_df = df[meta_cols].copy()

    for condition, reps in rep_dict.items():
        rep_cols = list(reps.values())

        if len(rep_cols) == 1:
            collapsed_df[condition] = df[rep_cols[0]]
        elif len(rep_cols) >= 2:
            values = df[rep_cols].to_numpy()
            closest_to_zero_idx = np.abs(values).argmin(axis=1)  # per-row index of min-|value| replicate
            collapsed_df[condition] = values[np.arange(len(values)), closest_to_zero_idx]

    return collapsed_df


def compute_lfc_zscore_and_save(df: pd.DataFrame, comparison: str, prefix: str, suffix: str = ""):
    """
    Run the full stats step for one comparison and write its three standard
    output files:
        <prefix>_LFC_rep<suffix>.csv       -- per-replicate log2 fold-change
        <prefix>_zscores_rep<suffix>.csv   -- per-replicate Z-scores
        <prefix>_zscores_min<suffix>.csv   -- replicates collapsed to one value

    Returns (log2fc_df, zscore_df, zscore_min_df).
    """
    log2fc_df, zscore_df = compute_log2fc_and_zscore(df, comparison)
    log2fc_df.to_csv(f"{prefix}_LFC_rep{suffix}.csv", index=False)
    zscore_df.to_csv(f"{prefix}_zscores_rep{suffix}.csv", index=False)
    print("\nNEW PIPELINE ZSCORE COLUMNS")
    print(zscore_df.columns.tolist())

    zscore_min_df = collapse_replicates_min(zscore_df)
    zscore_min_df.to_csv(f"{prefix}_zscores_min{suffix}.csv", index=False)

    return log2fc_df, zscore_df, zscore_min_df


# =============================================================================
# CoelhoNatGen study
# =============================================================================

def process_CoelhoNatGen(df: pd.DataFrame) -> pd.DataFrame:
    """
    Reshape raw CoelhoNatGen read counts (one row per guide, one column per raw
    condition string) into a wide table keyed by Guide/Gene/Editor/pDNA,
    with one cleanly-named column per (cell line, treatment, replicate)
    condition, then log2-normalize and pDNA-filter it.
    """
    # --- long format: one row per (guide, raw condition) --------------------
    df_melted = df.melt(id_vars=["guide", "Gene", "plasmid_reads"], var_name="Condition", value_name="Reads")
    df_melted["Gene"] = df_melted["Gene"].fillna("Unknown")

    # --- parse the raw condition string into its component parts -----------
    df_melted["Cell_Line"] = df_melted["Condition"].str.extract(r'^(H23|HT29|PC9|MHHES1)')
    df_melted["Editor"] = df_melted["Condition"].str.extract(r'_(CBE|ABE)_')
    df_melted["Experiment"] = df_melted["Condition"].str.extract(
        r'_(T0|Sotor|Control|Adag|Tram|Pict|DebCet|Gefit|Osim|Olap|Nirap|)_'
    )
    df_melted["Replicate"] = df_melted["Condition"].str.extract(r'_(exp\d+)_')

    # --- rebuild a clean condition label from the parsed parts --------------
    df_melted = df_melted.drop(columns=["Condition"])
    df_melted["Condition"] = df_melted["Cell_Line"] + "_" + df_melted["Experiment"] + "_" + df_melted["Replicate"]
    df_melted = df_melted.drop(columns=["Cell_Line", "Experiment", "Replicate"])

    # --- collapse any duplicate rows for the same guide/condition ----------
    df_melted = df_melted.groupby(["guide", "Gene", "Editor", "plasmid_reads", "Condition"], as_index=False).sum()

    # --- back to wide format: one column per condition ----------------------
    df_pivot = df_melted.pivot(
        index=["guide", "Gene", "Editor", "plasmid_reads"], columns="Condition", values="Reads"
    ).reset_index()
    df_pivot.columns.name = None

    # --- cosmetic renames: cell line codes, pDNA/Guide, replicate labels ----
    for pattern, replacement in CELL_LINE_RENAMES.items():
        df_pivot.columns = df_pivot.columns.str.replace(pattern, replacement, regex=True)

    df_pivot.rename(columns={'plasmid_reads': 'pDNA', 'guide': 'Guide'}, inplace=True)

    for old, new in REPLICATE_RENAMES.items():
        df_pivot.columns = [re.sub(old, new, col) for col in df_pivot.columns]

    df_pivot.columns = df_pivot.columns.str.replace("__", "_", regex=False)
    df_pivot["Gene"] = df_pivot["Gene"].replace("Unknown", pd.NA)

    # --- pull in the real Gene annotation for control guides ----------------
    # The raw counts file doesn't reliably label which guides are controls,
    # so we look them up from the samplesheet's guide library and overwrite
    # "Unknown" with the real control label where one exists.
    CoelhoNatGen_QC = _read_samplesheet('BE sgRNA library_MC')[['GeneID', 'sgRNA sequence', 'Editor']]
    CoelhoNatGen_QC = CoelhoNatGen_QC.rename(columns={'sgRNA sequence': 'Guide', 'GeneID': 'Gene'})
    CoelhoNatGen_QC["Gene"] = CoelhoNatGen_QC["Gene"].fillna("Unknown")
    CoelhoNatGen_QC['Editor'] = CoelhoNatGen_QC['Editor'].replace({'A-G': 'ABE', 'C-T': 'CBE'})
    CoelhoNatGen_QC = CoelhoNatGen_QC[CoelhoNatGen_QC['Gene'].isin(CoelhoNatGen_CONTROL_GENES)]  # keep only control guides

    CoelhoNatGen = df_pivot.merge(CoelhoNatGen_QC, on=["Guide", "Editor"], how='left')
    CoelhoNatGen["Gene_x"] = CoelhoNatGen.apply(
        lambda row: row["Gene_y"] if pd.notna(row["Gene_y"]) else row["Gene_x"], axis=1
    )
    CoelhoNatGen = CoelhoNatGen.drop(columns=["Gene_y"]).rename(columns={'Gene_x': 'Gene'})

    return poola_normalize_and_filter(CoelhoNatGen, pdna_col="pDNA", meta_cols=META_COLS)


# =============================================================================
# dt study
# =============================================================================

def _load_and_normalize_dt(path_or_buf) -> pd.DataFrame:
    """Read one dt counts CSV, coerce its read columns to numeric, and run
    the standard normalize + pDNA-filter step."""
    df = pd.read_csv(path_or_buf).drop(columns='Unnamed: 0')
    numeric_cols = df.columns.difference(['Construct Barcode'])
    df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')
    return poola_normalize_and_filter(df, pdna_col='pDNA_CP2165', meta_cols=['Construct Barcode'])


def _merge_cell_line(
    do_df,
    lin_df,
    do_cols,
    lin_cols,
    do_suffix,
    lin_suffix
):
    """
    Merge the DO/PIC and LIN/SCH tables for one cell line.

    The explicit suffixes are important because the downstream
    Comparisons sheet expects the same column names that were produced
    by the old pipeline.
    """

    # Keep only the columns relevant for this cell-line comparison.
    # Construct Barcode is retained so the two tables can be joined.
    do_subset = do_df[do_cols].copy()
    lin_subset = lin_df[lin_cols].copy()

    # Merge DO/PIC and LIN/SCH on the construct barcode.
    #
    # IMPORTANT:
    # Use cell-line-specific suffixes here. The old pipeline used:
    #   A375 -> _A375_DO_PIC / _A375_SCH_LIN
    #   HT29 -> _HT29_DO_PIC / _HT29_SCH_LIN
    #
    # This preserves the column names expected by the Comparisons sheet.
    merged = do_subset.merge(
        lin_subset,
        on='Construct Barcode',
        suffixes=(do_suffix, lin_suffix)
    )

    # Rename the construct barcode to the common identifier used
    # throughout the downstream pipeline.
    merged = merged.rename(
        columns={'Construct Barcode': 'Guide'}
    )

    # All dt samples in this pipeline are CBE samples.
    merged['Editor'] = 'CBE'

    return merged


def process_dt(link1, link2, comparison: str):
    """
    Build the dt study's combined table across both cell lines (A375, HT29)
    and both treatment sets (DO/PIC, SCH/LIN), annotate it with Gene labels,
    then run the LFC/Z-score/replicate-collapse step for `comparison`.
    """
    # A375 comes from caller-supplied paths; HT29 files are fixed for this study.
    A375_DO = _load_and_normalize_dt(link1)
    A375_LIN = _load_and_normalize_dt(link2)
    HT29_DO = _load_and_normalize_dt(dt_HT29_DO_PATH)
    HT29_LIN = _load_and_normalize_dt(dt_HT29_LIN_PATH)

    A375_merge = _merge_cell_line(A375_DO,A375_LIN,A375_DO_COLS,A375_LIN_COLS,'_A375_DO_PIC','_A375_SCH_LIN')
    HT29_merge = _merge_cell_line(HT29_DO,HT29_LIN,HT29_DO_COLS,HT29_LIN_COLS,'_HT29_DO_PIC','_HT29_SCH_LIN')

    dt_merge = HT29_merge.merge(A375_merge, on='Guide', suffixes=('_DO_PIC', '_SCH_LIN'))
    dt_merge = dt_merge.drop(columns='Editor_DO_PIC').rename(columns={'Editor_SCH_LIN': 'Editor'})

    print("dt_merge:", len(dt_merge.columns), dt_merge.columns.tolist())

    processed = dt_merge.copy()  # snapshot before Gene annotation, saved as-is below

    # --- attach Gene labels from the dt guide library -----------------------
    bes_dt = _read_samplesheet('BE sgRNA library_EG')
    bes_dt = bes_dt.assign(Editor=bes_dt['Editor'].replace({'A-G': 'ABE', 'C-T': 'CBE'}))
    bes_dt = bes_dt[['sgRNA sequence', 'Editor', 'GeneID']].rename(columns={'sgRNA sequence': 'Guide', 'GeneID': 'Gene'})
    processed.to_csv('dt_processed.csv', index=None)

    dt_merge = dt_merge.merge(bes_dt, on=['Guide', 'Editor'], how='left')
    dt_merge = reorder_columns(dt_merge)

    if comparison in dt_COMPARISON_SUFFIXES:
        df_lfc, df_zscores, zscores_min = compute_lfc_zscore_and_save(
            dt_merge, comparison, "dt", dt_COMPARISON_SUFFIXES[comparison]
        )

    return processed, df_lfc, df_zscores, zscores_min


# =============================================================================
# Entry point
# =============================================================================

def process_df(file1, study: str, file2=None, sheet_name=None, comparison: str = None):
    """
    Single public entry point for both studies.

    study='CoelhoNatGen': file1 = path to the raw CoelhoNatGen counts CSV.
    study='dt': file1, file2 = paths to the A375 DO/PIC and SCH/LIN counts
                CSVs (HT29's paths are fixed, see dt_HT29_DO_PATH /
                dt_HT29_LIN_PATH above).

    `comparison`, if given, must be a key in CoelhoNatGen_COMPARISON_SUFFIXES (for
    study='CoelhoNatGen') or dt_COMPARISON_SUFFIXES (for study='dt'); it selects which
    LFC/Z-score CSVs get computed and saved.

    Returns (df, lfc_data, zscores, zscores_min). `zscores_min` (and, for
    study='dt', `lfc_data`/`zscores`) stay unset if `comparison` doesn't
    trigger a matching branch -- see the NOTE in `process_dt`.
    """
    zscores_min = None

    assert study in VALID_STUDIES, f"Invalid study: {study}. Must be one of {VALID_STUDIES}"
    if comparison is not None:
        assert comparison in VALID_COMPARISONS, f"Invalid comparison: {comparison}. Must be one of {VALID_COMPARISONS}"

    if study == 'CoelhoNatGen':
        df = pd.read_csv(f'{file1}').sort_values(by='guide')
        df = process_CoelhoNatGen(df)
        df.to_csv("CoelhoNatGen_processed.csv", index=False)

        if comparison in CoelhoNatGen_COMPARISON_SUFFIXES:
            lfc_data, zscores, zscores_min = compute_lfc_zscore_and_save(
                df, comparison, "CoelhoNatGen", CoelhoNatGen_COMPARISON_SUFFIXES[comparison]
            )

    if study == 'dt':
        df, lfc_data, zscores, zscores_min = process_dt(file1, file2, comparison)

    return df, lfc_data, zscores, zscores_min

In [3]:
df_CoelhoNatGen,lfc_data_CoelhoNatGen,zscores_CoelhoNatGen,zscores_min_CoelhoNatGen=process_df('MC_readcouns.csv','CoelhoNatGen',comparison='CoelhoNatGen_processed')

/tmp/ipykernel_1050/2351599442.py:496: DtypeWarning: Columns (76,77,78) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f'{file1}').sort_values(by='guide')



NEW PIPELINE ZSCORE COLUMNS
['Guide', 'Gene', 'Editor', 'H_23_Adag_Control_RepA', 'H_23_Adag_Control_RepB', 'H_23_Sotor_Control_RepA', 'H_23_Sotor_Control_RepB', 'H_23_T0_Control_RepA', 'H_23_T0_Control_RepB', 'HT_29_Deb_Cet_Control_RepA', 'HT_29_Deb_Cet_Control_RepB', 'HT_29_Pict_Control_RepA', 'HT_29_Pict_Control_RepB', 'HT_29_T0_Control_RepA', 'HT_29_T0_Control_RepB', 'HT_29_Tram_Control_RepA', 'HT_29_Tram_Control_RepB', 'PC_9_Gefit_Control_RepA', 'PC_9_Gefit_Control_RepB', 'PC_9_Osim_Control_RepA', 'PC_9_Osim_Control_RepB', 'PC_9_T0_Control_RepA', 'PC_9_T0_Control_RepB', 'MHHES_1_Nirap_Control_RepA', 'MHHES_1_Nirap_Control_RepB', 'MHHES_1_Olap_Control_RepA', 'MHHES_1_Olap_Control_RepB', 'MHHES_1_T0_Control_RepA', 'MHHES_1_T0_Control_RepB', 'H_23_Adag_Plasmid_RepA', 'H_23_Adag_Plasmid_RepB', 'H_23_Sotor_Plasmid_RepA', 'H_23_Sotor_Plasmid_RepB', 'H_23_T0_Plasmid_RepA', 'H_23_T0_Plasmid_RepB', 'HT_29_Deb_Cet_Plasmid_RepA', 'HT_29_Deb_Cet_Plasmid_RepB', 'HT_29_Pict_Plasmid_RepA', 'HT_

In [4]:
df,lfc_data,zscores , zscores_min=process_df('counts_with_mismatches_CBE_activity_A375_DO_PIC.csv','dt',
           'counts_with_mismatches_CBE_activity_A375_LIN_SCH.csv',
           comparison='dt')

dt_merge: 22 ['Guide', 'HT29_RDA270_CP2165_PIC_RepA', 'HT29_RDA270_CP2165_DO_RepB', 'HT29_RDA270_CP2165_DO_RepA', 'HT29_RDA270_CP2165_PIC_RepB', 'pDNA_CP2165_HT29_DO_PIC', 'HT29_RDA270_CP2165_SCH_RepB', 'pDNA_CP2165_HT29_SCH_LIN', 'HT29_RDA270_CP2165_LIN_RepB', 'HT29_RDA270_CP2165_SCH_RepA', 'HT29_RDA270_CP2165_LIN_RepA', 'A375_RDA270_CP2165_PIC_RepA', 'A375_RDA270_CP2165_DO_RepB', 'A375_RDA270_CP2165_DO_RepA', 'A375_RDA270_CP2165_PIC_RepB', 'pDNA_CP2165_A375_DO_PIC', 'A375_RDA270_CP2165_SCH_RepB', 'pDNA_CP2165_A375_SCH_LIN', 'A375_RDA270_CP2165_LIN_RepB', 'A375_RDA270_CP2165_SCH_RepA', 'A375_RDA270_CP2165_LIN_RepA', 'Editor']

NEW PIPELINE ZSCORE COLUMNS
['Guide', 'Gene', 'Editor', 'A375_PIC_Control_RepA', 'A375_PIC_Control_RepB', 'A375_SCH_Control_RepA', 'A375_SCH_Control_RepB', 'A375_LIN_Control_RepA', 'A375_LIN_Control_RepB', 'HT29_PIC_Control_RepA', 'HT29_PIC_Control_RepB', 'HT29_SCH_Control_RepA', 'HT29_SCH_Control_RepB', 'HT29_LIN_Control_RepA', 'HT29_LIN_Control_RepB', 'A375_PI

Here we will have the script that does the merge into 2 datasets to Analyse. One is drug-treated condition relative to the Plasmid population. The other is drug-treated condition relative to the Control population

In [5]:
# ============================================================
# 1. LOAD Z-SCORE DATA
# ============================================================

data = pd.read_csv('dt_zscores_min.csv')
CoelhoNatGen = pd.read_csv('CoelhoNatGen_zscores_min.csv')


# ============================================================
# 2. SELECT PLASMID / CONTROL COLUMNS
# ============================================================

def select_condition_columns(df, condition):
    """
    Keep Guide/Gene/Editor plus the requested condition columns.

    Plasmid:
        Keep every column containing 'Plasmid'.

    Control:
        Keep columns containing 'Control', but exclude columns
        containing 'Plasmid'.

    This prevents columns such as:
        H_23_Control_Plasmid
    from accidentally being included in the Control dataset.
    """

    metadata_cols = ['Guide', 'Gene', 'Editor']

    if condition == 'Plasmid':
        condition_cols = [
            col for col in df.columns
            if 'Plasmid' in col
        ]

    elif condition == 'Control':
        condition_cols = [
            col for col in df.columns
            if 'Control' in col and 'Plasmid' not in col
        ]

    else:
        raise ValueError(
            f"Unknown condition: {condition}. "
            f"Use 'Plasmid' or 'Control'."
        )

    return df[metadata_cols + condition_cols].copy()


# Plasmid datasets
data_plasmid = select_condition_columns(data, 'Plasmid')
CoelhoNatGen_plasmid = select_condition_columns(CoelhoNatGen, 'Plasmid')

# Control datasets
data_control = select_condition_columns(data, 'Control')
CoelhoNatGen_control = select_condition_columns(CoelhoNatGen, 'Control')


# ============================================================
# 3. ADD SOURCE LABELS
# ============================================================

data_plasmid['source'] = 'data'
CoelhoNatGen_plasmid['source'] = 'CoelhoNatGen'

data_control['source'] = 'data'
CoelhoNatGen_control['source'] = 'CoelhoNatGen'


# ============================================================
# 4. REMOVE T0 COLUMNS
# ============================================================

# Remove T0 Plasmid columns from CoelhoNatGen
CoelhoNatGen_plasmid = CoelhoNatGen_plasmid.drop(
    columns=[
        'H_23_T0_Plasmid',
        'HT_29_T0_Plasmid',
        'PC_9_T0_Plasmid',
        'MHHES_1_T0_Plasmid'
    ],
    errors='ignore'
)

# Remove T0 Control columns from CoelhoNatGen
CoelhoNatGen_control = CoelhoNatGen_control.drop(
    columns=[
        'H_23_T0_Control',
        'HT_29_T0_Control',
        'PC_9_T0_Control',
        'MHHES_1_T0_Control'
    ],
    errors='ignore'
)


# ============================================================
# 5. RENAME COELHONATGEN PLASMID COLUMNS
# ============================================================

CoelhoNatGen_plasmid = CoelhoNatGen_plasmid.rename(
    columns={
        'H_23_Adag_Plasmid': 'H23_ADAG_Plasmid',
        'H_23_Sotor_Plasmid': 'H23_SOTOR_Plasmid',
        'HT_29_Deb_Cet_Plasmid': 'HT29_DEB-CET_Plasmid',
        'HT_29_Pict_Plasmid': 'HT29_PIC_Plasmid_CoelhoNatGen',
        'HT_29_Tram_Plasmid': 'HT29_TRAM_Plasmid',
        'PC_9_Gefit_Plasmid': 'PC9_GEFIT_Plasmid',
        'PC_9_Osim_Plasmid': 'PC9_OSIM_Plasmid',
        'MHHES_1_Nirap_Plasmid': 'MHHES1_NIRAP_Plasmid',
        'MHHES_1_Olap_Plasmid': 'MHHES1_OLAP_Plasmid',

        # Control-treatment plasmid columns
        'H_23_Control_Plasmid': 'H23_Control_Plasmid',
        'HT_29_Control_Plasmid': 'HT29_Control_Plasmid',
        'PC_9_Control_Plasmid': 'PC9_Control_Plasmid',
        'MHHES_1_Control_Plasmid': 'MHHES1_Control_Plasmid'
    }
)


# ============================================================
# 6. RENAME DATA PLASMID COLUMNS
# ============================================================

data_plasmid = data_plasmid.rename(
    columns={
        'HT29_PIC_Plasmid': 'HT29_PIC_Plasmid_data'
    }
)


# ============================================================
# 7. RENAME COELHONATGEN CONTROL COLUMNS
# ============================================================

CoelhoNatGen_control = CoelhoNatGen_control.rename(
    columns={
        'H_23_Adag_Control': 'H23_ADAG_Control',
        'H_23_Sotor_Control': 'H23_SOTOR_Control',
        'HT_29_Deb_Cet_Control': 'HT29_DEB-CET_Control',
        'HT_29_Pict_Control': 'HT29_PIC_Control_CoelhoNatGen',
        'HT_29_Tram_Control': 'HT29_TRAM_Control',
        'PC_9_Gefit_Control': 'PC9_GEFIT_Control',
        'PC_9_Osim_Control': 'PC9_OSIM_Control',
        'MHHES_1_Nirap_Control': 'MHHES1_NIRAP_Control',
        'MHHES_1_Olap_Control': 'MHHES1_OLAP_Control',

        # Control-condition columns, if present
        'H_23_Control_Control': 'H23_Control_Control',
        'HT_29_Control_Control': 'HT29_Control_Control',
        'PC_9_Control_Control': 'PC9_Control_Control',
        'MHHES_1_Control_Control': 'MHHES1_Control_Control'
    }
)


# ============================================================
# 8. RENAME DATA CONTROL COLUMNS
# ============================================================

data_control = data_control.rename(
    columns={
        'HT29_PIC_Control': 'HT29_PIC_Control_data'
    }
)


# ============================================================
# 9. CHECK COLUMNS BEFORE MERGING
# ============================================================

print("\n" + "=" * 70)
print("PLASMID COLUMNS")
print("=" * 70)

print("\nData:")
print(data_plasmid.columns.tolist())

print("\nCoelhoNatGen:")
print(CoelhoNatGen_plasmid.columns.tolist())


print("\n" + "=" * 70)
print("CONTROL COLUMNS")
print("=" * 70)

print("\nData:")
print(data_control.columns.tolist())

print("\nCoelhoNatGen:")
print(CoelhoNatGen_control.columns.tolist())


# ============================================================
# 10. MERGE PLASMID DATA
# ============================================================

plasmid_data = data_plasmid.merge(
    CoelhoNatGen_plasmid,
    on=['Guide', 'Gene', 'Editor'],
    how='outer'
)

plasmid_data = (
    plasmid_data
    .sort_values('Guide')
    .reset_index(drop=True)
)


# ============================================================
# 11. MERGE CONTROL DATA
# ============================================================

control_data = data_control.merge(
    CoelhoNatGen_control,
    on=['Guide', 'Gene', 'Editor'],
    how='outer'
)

control_data = (
    control_data
    .sort_values('Guide')
    .reset_index(drop=True)
)


# ============================================================
# 12. RESOLVE SOURCE FOR PLASMID DATA
# ============================================================

plasmid_data['source'] = np.select(
    [
        plasmid_data['source_x'].eq('CoelhoNatGen') &
        plasmid_data['source_y'].isna(),

        plasmid_data['source_x'].eq('data') &
        plasmid_data['source_y'].isna(),

        plasmid_data['source_x'].eq('data') &
        plasmid_data['source_y'].eq('CoelhoNatGen'),

        plasmid_data['source_x'].isna() &
        plasmid_data['source_y'].eq('CoelhoNatGen')
    ],
    [
        'CoelhoNatGen',
        'data',
        'data_CoelhoNatGen',
        'CoelhoNatGen'
    ],
    default=None
)

plasmid_data = plasmid_data.drop(
    columns=['source_x', 'source_y']
)


# ============================================================
# 13. RESOLVE SOURCE FOR CONTROL DATA
# ============================================================

control_data['source'] = np.select(
    [
        control_data['source_x'].eq('CoelhoNatGen') &
        control_data['source_y'].isna(),

        control_data['source_x'].eq('data') &
        control_data['source_y'].isna(),

        control_data['source_x'].eq('data') &
        control_data['source_y'].eq('CoelhoNatGen'),

        control_data['source_x'].isna() &
        control_data['source_y'].eq('CoelhoNatGen')
    ],
    [
        'CoelhoNatGen',
        'data',
        'data_CoelhoNatGen',
        'CoelhoNatGen'
    ],
    default=None
)

control_data = control_data.drop(
    columns=['source_x', 'source_y']
)


# ============================================================
# 14. CHECK FINAL MERGED DATA
# ============================================================

print("\n" + "=" * 70)
print("FINAL PLASMID DATA")
print("=" * 70)

print("Shape:", plasmid_data.shape)

print("\nPlasmid columns:")
print([
    col for col in plasmid_data.columns
    if 'Plasmid' in col
])


print("\n" + "=" * 70)
print("FINAL CONTROL DATA")
print("=" * 70)

print("Shape:", control_data.shape)

print("\nControl columns:")
print([
    col for col in control_data.columns
    if 'Control' in col
])




PLASMID COLUMNS

Data:
['Guide', 'Gene', 'Editor', 'A375_PIC_Plasmid', 'A375_SCH_Plasmid', 'A375_LIN_Plasmid', 'HT29_PIC_Plasmid_data', 'HT29_SCH_Plasmid', 'HT29_LIN_Plasmid', 'A375_DO_Plasmid', 'HT29_DO_Plasmid', 'source']

CoelhoNatGen:
['Guide', 'Gene', 'Editor', 'H23_ADAG_Plasmid', 'H23_SOTOR_Plasmid', 'HT29_DEB-CET_Plasmid', 'HT29_PIC_Plasmid_CoelhoNatGen', 'HT29_TRAM_Plasmid', 'PC9_GEFIT_Plasmid', 'PC9_OSIM_Plasmid', 'MHHES1_NIRAP_Plasmid', 'MHHES1_OLAP_Plasmid', 'H23_Control_Plasmid', 'HT29_Control_Plasmid', 'PC9_Control_Plasmid', 'MHHES1_Control_Plasmid', 'source']

CONTROL COLUMNS

Data:
['Guide', 'Gene', 'Editor', 'A375_PIC_Control', 'A375_SCH_Control', 'A375_LIN_Control', 'HT29_PIC_Control_data', 'HT29_SCH_Control', 'HT29_LIN_Control', 'source']

CoelhoNatGen:
['Guide', 'Gene', 'Editor', 'H23_ADAG_Control', 'H23_SOTOR_Control', 'HT29_DEB-CET_Control', 'HT29_PIC_Control_CoelhoNatGen', 'HT29_TRAM_Control', 'PC9_GEFIT_Control', 'PC9_OSIM_Control', 'MHHES1_NIRAP_Control', 'MHHE

Hit assignment on Plasmid data

In [9]:
from scipy.stats import norm
from statsmodels.stats.multitest import multipletests
import numpy as np


from scipy.stats import norm
from statsmodels.stats.multitest import multipletests
import numpy as np

X=plasmid_data.copy()
exclude_cols = ['Guide', 'Gene', 'Editor','source']
zscore_cols = [col for col in X.columns if col not in exclude_cols]

for cond in zscore_cols:
    print(f"Processing condition: {cond}")

    all_z = X[cond]
    neg_ctrl_z = X.loc[X['Gene'] == 'Intergenic control', cond]

    mu = neg_ctrl_z.mean(skipna=True)
    sigma = neg_ctrl_z.std(skipna=True)

    # Initialize pvals and fdrs with NaNs
    pvals = np.full_like(all_z, np.nan, dtype=np.float64)
    fdrs = np.full_like(all_z, np.nan, dtype=np.float64)

    # Indices where all_z is not NaN
    valid_idx = all_z.notna()

    # Compute p-values only for valid entries
    z_scores = (all_z[valid_idx] - mu) / sigma
    pvals_valid = 2 * norm.sf(np.abs(z_scores))

    # FDR correction
    _, fdrs_valid, _, _ = multipletests(pvals_valid, alpha=0.05, method='fdr_bh')

    # Assign back to full arrays
    pvals[valid_idx] = pvals_valid
    fdrs[valid_idx] = fdrs_valid

    # Add results to dataframe
    X[f"{cond}_pval"] = pvals
    X[f"{cond}_fdr"] = fdrs

    print(f"Condition: {cond}")
    print(f"Negative controls count: {neg_ctrl_z.count()}")
    print(f"Negative control mean: {mu}, std: {sigma}")
    print(f"Any NaN in pvals? {np.isnan(pvals).any()}")
    print(f"All finite pvals? {np.isfinite(pvals[valid_idx]).all()}")


Processing condition: A375_PIC_Plasmid
Condition: A375_PIC_Plasmid
Negative controls count: 991
Negative control mean: -0.05768429391959584, std: 0.5726418025142905
Any NaN in pvals? True
All finite pvals? True
Processing condition: A375_SCH_Plasmid
Condition: A375_SCH_Plasmid
Negative controls count: 991
Negative control mean: -0.15671502665601425, std: 0.566045011660789
Any NaN in pvals? True
All finite pvals? True
Processing condition: A375_LIN_Plasmid
Condition: A375_LIN_Plasmid
Negative controls count: 991
Negative control mean: -0.020114884890758033, std: 0.6144342511480272
Any NaN in pvals? True
All finite pvals? True
Processing condition: HT29_PIC_Plasmid_data
Condition: HT29_PIC_Plasmid_data
Negative controls count: 991
Negative control mean: 0.0021717307615956314, std: 0.6995807387636294
Any NaN in pvals? True
All finite pvals? True
Processing condition: HT29_SCH_Plasmid
Condition: HT29_SCH_Plasmid
Negative controls count: 991
Negative control mean: -0.03514955715671299, std:

In [10]:
def classify_hit(z, fdr, threshold=0.05):
    if fdr < threshold:
        if z > 0:
            return "positive"
        elif z < 0:
            return "negative"
    return "non-hit"

for cond in zscore_cols:
    z_col = cond
    fdr_col = f"{cond}_fdr"
    label_col = f"{cond}_hit"

    X[label_col] = X.apply(
        lambda row: classify_hit(row[z_col], row[fdr_col]),
        axis=1
    )

In [11]:
df_plasmid=X.copy()

In [12]:
df_plasmid.to_csv('df_plasmid.csv',index=None)

Hit assignemnt for control data

In [13]:
from scipy.stats import norm
from statsmodels.stats.multitest import multipletests
import numpy as np


from scipy.stats import norm
from statsmodels.stats.multitest import multipletests
import numpy as np

X=control_data.copy()
exclude_cols = ['Guide', 'Gene', 'Editor','source']
zscore_cols = [col for col in X.columns if col not in exclude_cols]

for cond in zscore_cols:
    print(f"Processing condition: {cond}")

    all_z = X[cond]
    neg_ctrl_z = X.loc[X['Gene'] == 'Intergenic control', cond]

    mu = neg_ctrl_z.mean(skipna=True)
    sigma = neg_ctrl_z.std(skipna=True)

    # Initialize pvals and fdrs with NaNs
    pvals = np.full_like(all_z, np.nan, dtype=np.float64)
    fdrs = np.full_like(all_z, np.nan, dtype=np.float64)

    # Indices where all_z is not NaN
    valid_idx = all_z.notna()

    # Compute p-values only for valid entries
    z_scores = (all_z[valid_idx] - mu) / sigma
    pvals_valid = 2 * norm.sf(np.abs(z_scores))

    # FDR correction
    _, fdrs_valid, _, _ = multipletests(pvals_valid, alpha=0.05, method='fdr_bh')

    # Assign back to full arrays
    pvals[valid_idx] = pvals_valid
    fdrs[valid_idx] = fdrs_valid

    # Add results to dataframe
    X[f"{cond}_pval"] = pvals
    X[f"{cond}_fdr"] = fdrs

    print(f"Condition: {cond}")
    print(f"Negative controls count: {neg_ctrl_z.count()}")
    print(f"Negative control mean: {mu}, std: {sigma}")
    print(f"Any NaN in pvals? {np.isnan(pvals).any()}")
    print(f"All finite pvals? {np.isfinite(pvals[valid_idx]).all()}")


Processing condition: A375_PIC_Control
Condition: A375_PIC_Control
Negative controls count: 991
Negative control mean: -0.06958373731011161, std: 0.5217998807817087
Any NaN in pvals? True
All finite pvals? True
Processing condition: A375_SCH_Control
Condition: A375_SCH_Control
Negative controls count: 991
Negative control mean: -0.1273270125501963, std: 0.5733103594709223
Any NaN in pvals? True
All finite pvals? True
Processing condition: A375_LIN_Control
Condition: A375_LIN_Control
Negative controls count: 991
Negative control mean: 0.014529726688955224, std: 0.593562126011116
Any NaN in pvals? True
All finite pvals? True
Processing condition: HT29_PIC_Control_data
Condition: HT29_PIC_Control_data
Negative controls count: 991
Negative control mean: 0.008590849856131913, std: 0.59798063498052
Any NaN in pvals? True
All finite pvals? True
Processing condition: HT29_SCH_Control
Condition: HT29_SCH_Control
Negative controls count: 991
Negative control mean: 0.0008463461306438328, std: 0.5

In [14]:
def classify_hit(z, fdr, threshold=0.05):
    if fdr < threshold:
        if z > 0:
            return "positive"
        elif z < 0:
            return "negative"
    return "non-hit"

for cond in zscore_cols:
    z_col = cond
    fdr_col = f"{cond}_fdr"
    label_col = f"{cond}_hit"

    X[label_col] = X.apply(
        lambda row: classify_hit(row[z_col], row[fdr_col]),
        axis=1
    )

In [15]:
df_control=X.copy()

In [16]:
df_control.to_csv('df_control.csv',index=None)

In [ ]:
df_plasmid

,Guide,Gene,Editor,A375_PIC_Plasmid,A375_SCH_Plasmid,A375_LIN_Plasmid,HT29_PIC_Plasmid_data,HT29_SCH_Plasmid,HT29_LIN_Plasmid,A375_DO_Plasmid,...,HT29_PIC_Plasmid_CoelhoNatGen_hit,HT29_TRAM_Plasmid_hit,PC9_GEFIT_Plasmid_hit,PC9_OSIM_Plasmid_hit,MHHES1_NIRAP_Plasmid_hit,MHHES1_OLAP_Plasmid_hit,H23_Control_Plasmid_hit,HT29_Control_Plasmid_hit,PC9_Control_Plasmid_hit,MHHES1_Control_Plasmid_hit
0,AAAAAAAAAAGTAAGCTTCA,KRAS,ABE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit
1,AAAAAAAAAAGTAAGCTTCA,KRAS,CBE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit
2,AAAAAAAAAATCAATGGTGG,BCL2,ABE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit
3,AAAAAAAAAATCAATGGTGG,BCL2,CBE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit
4,AAAAAAAAAATTAATGTCTT,KRAS,ABE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77003,TTTTGTGCCCTGGACAAAGT,BRAF,CBE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit
77004,TTTTGTGTGACAAAGCGTAA,EGFR,ABE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit
77005,TTTTGTGTGACAAAGCGTAA,EGFR,CBE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit
77006,TTTTGTTTCCTTGCTCACTG,BRAF,ABE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit


In [ ]:
df_control

,Guide,Gene,Editor,A375_PIC_Control,A375_SCH_Control,A375_LIN_Control,HT29_PIC_Control_data,HT29_SCH_Control,HT29_LIN_Control,H23_ADAG_Control,...,HT29_LIN_Control_hit,H23_ADAG_Control_hit,H23_SOTOR_Control_hit,HT29_DEB-CET_Control_hit,HT29_PIC_Control_CoelhoNatGen_hit,HT29_TRAM_Control_hit,PC9_GEFIT_Control_hit,PC9_OSIM_Control_hit,MHHES1_NIRAP_Control_hit,MHHES1_OLAP_Control_hit
0,AAAAAAAAAAGTAAGCTTCA,KRAS,ABE,NaN,NaN,NaN,NaN,NaN,NaN,0.403765,...,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit
1,AAAAAAAAAAGTAAGCTTCA,KRAS,CBE,NaN,NaN,NaN,NaN,NaN,NaN,-0.334297,...,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit
2,AAAAAAAAAATCAATGGTGG,BCL2,ABE,NaN,NaN,NaN,NaN,NaN,NaN,-0.032180,...,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit
3,AAAAAAAAAATCAATGGTGG,BCL2,CBE,NaN,NaN,NaN,NaN,NaN,NaN,0.227702,...,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit
4,AAAAAAAAAATTAATGTCTT,KRAS,ABE,NaN,NaN,NaN,NaN,NaN,NaN,-0.309726,...,non-hit,non-hit,non-hit,negative,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77003,TTTTGTGCCCTGGACAAAGT,BRAF,CBE,NaN,NaN,NaN,NaN,NaN,NaN,-0.120373,...,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit
77004,TTTTGTGTGACAAAGCGTAA,EGFR,ABE,NaN,NaN,NaN,NaN,NaN,NaN,0.363145,...,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit
77005,TTTTGTGTGACAAAGCGTAA,EGFR,CBE,NaN,NaN,NaN,NaN,NaN,NaN,0.494727,...,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit
77006,TTTTGTTTCCTTGCTCACTG,BRAF,ABE,NaN,NaN,NaN,NaN,NaN,NaN,0.056710,...,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit,non-hit


# Screen QC

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from scipy.ndimage import gaussian_filter

# ---------------- settings ----------------
INFILE = "dt_zscores_rep.csv"
SEP = None                             # None = auto-detect (comma or tab)
OUTFILE = "replicate_correlations"     # saves .png and .pdf

CELL_LINES = ["A375", "HT29"]                                   # rows (top -> bottom)
DRUGS = [("PIC", "Pictilisib"),                                 # columns (left -> right)
         ("LIN", "Linsitinib"),
         ("SCH", "SCH772984")]

XLABEL = "Replicate A z-score"
YLABEL = "Replicate B z-score"
LIM = (-8, 8)

DOT_SIZE = 8
LABEL_SIZE = 13                        # axis labels (x label / y label)
TICK_SIZE = 10                         # tick numbers
TITLE_SIZE = 13                        # drug names
R_SIZE = 12                            # r=... text
CELL_SIZE = 15                         # A375 / HT29 row labels
CELL_X = -0.45                         # horizontal position of cell line labels
# -----------------------------------------

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Roboto", "Arial", "DejaVu Sans"],
    "pdf.fonttype": 42,                # editable text in Illustrator
})


def density_colors(x, y, bins=200, sigma=2):
    """Fast KDE-like density per point via a smoothed 2D histogram."""
    h, xe, ye = np.histogram2d(x, y, bins=bins)
    h = gaussian_filter(h, sigma=sigma)
    xi = np.clip(np.digitize(x, xe) - 1, 0, bins - 1)
    yi = np.clip(np.digitize(y, ye) - 1, 0, bins - 1)
    return h[xi, yi]


df = pd.read_csv(INFILE, sep=SEP, engine="python")

fig, axes = plt.subplots(2, 3, figsize=(9.5, 6.2))

for r, cell in enumerate(CELL_LINES):
    for c, (code, name) in enumerate(DRUGS):
        ax = axes[r, c]
        a = df[f"{cell}_{code}_Control_RepA"]
        b = df[f"{cell}_{code}_Control_RepB"]

        m = a.notna() & b.notna() & np.isfinite(a) & np.isfinite(b)
        x, y = a[m].to_numpy(), b[m].to_numpy()

        # colour by density, plot densest points last so they sit on top
        z = density_colors(x, y)
        order = z.argsort()
        ax.scatter(x[order], y[order], c=z[order], cmap="viridis",
                   s=DOT_SIZE, linewidths=0, alpha=0.8, rasterized=True)

        rval, _ = pearsonr(x, y)
        ax.text(0.03, 0.97, f"r={rval:.2f}", transform=ax.transAxes,
                ha="left", va="top", fontsize=R_SIZE)

        ax.set_xlim(LIM)
        ax.set_ylim(LIM)
        ax.set_xticks(np.arange(-15, 20, 5))
        ax.set_yticks(np.arange(-15, 20, 5))
        ax.tick_params(labelsize=TICK_SIZE, length=3, width=0.7)
        ax.spines[["top", "right"]].set_visible(False)
        for s in ("left", "bottom"):
            ax.spines[s].set_linewidth(0.7)

        # drug name on top row only
        if r == 0:
            ax.set_title(name, fontsize=TITLE_SIZE, pad=8)

        # x label on bottom row only
        if r == len(CELL_LINES) - 1:
            ax.set_xlabel(XLABEL, fontsize=LABEL_SIZE)

        # y label + cell line annotation on the first column of each row
        if c == 0:
            ax.set_ylabel(YLABEL, fontsize=LABEL_SIZE)
            ax.annotate(cell, xy=(CELL_X, 0.5), xycoords="axes fraction",
                        rotation=90, ha="center", va="center",
                        fontsize=CELL_SIZE, fontweight="bold")

fig.tight_layout(w_pad=1.5, h_pad=1.2, rect=[0.03, 0, 1, 1])
fig.savefig(f"{OUTFILE}.png", dpi=300, bbox_inches="tight")
fig.savefig(f"{OUTFILE}.pdf", bbox_inches="tight")
plt.show()

# Off-target filtering

Off-target prediction for the guides used in the dt screen was performed using FlashFry
https://github.com/mckennalab/FlashFry
against the human genome (hg38), considering  all possible SpG-Cas9 NGN PAMs.

All guides from the dt library were mapped against potential genomic target sites, and the number of matches containing 1, 2, 3, or 4 mismatches was recorded.
The resulting off-target analysis is provided in EG_off-targets.xlsx.

To minimize potential confounding effects from off-target activity, the final z-score analysis was restricted to guides with exactly one perfect genomic match,
corresponding to their intended on-target site. Guides with additional perfect matches elsewhere in the genome were excluded from the filtered dataset.

In [ ]:
# ---------- 1. Load the two original files ----------
offt = pd.read_excel("EG_off-targets.xlsx")
zscores = pd.read_csv("dt_zscores_min.csv")

# Clean up column names and Guide values (stray whitespace can break matching)
offt.columns = offt.columns.str.strip()
zscores.columns = zscores.columns.str.strip()
offt["Guide"] = offt["Guide"].astype(str).str.strip()
zscores["Guide"] = zscores["Guide"].astype(str).str.strip()

# ---------- 2. Filter off-targets: keep OT_0mm of 0 or 1 ----------
offt["OT_0mm"] = pd.to_numeric(offt["OT_0mm"], errors="coerce")
offt_filtered = offt[offt["OT_0mm"].isin([0, 1])]

print(f"Off-target file: {len(offt)} rows -> {len(offt_filtered)} rows after OT_0mm filter")


# ---------- 3. Merge with the zscores file ----------
# One row per guide so the merge can't duplicate zscore rows
offt_unique = offt_filtered[["Guide", "OT_0mm"]].drop_duplicates(subset="Guide")

n_before = len(zscores)
merged = zscores.merge(offt_unique, on="Guide", how="inner")  # keeps only guides in the filtered file
n_after = len(merged)

# ---------- 4. Report row counts ----------
print(f"\ndt_zscores_min.csv")
print(f"Rows before merge: {n_before}")
print(f"Rows after merge:  {n_after}")
print(f"Rows removed:      {n_before - n_after}")

# ---------- 5. Save the final result ----------
merged.to_csv("dt_zscores_min_off-targets-filtered.csv", index=False)